In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [2]:
from setproctitle import setproctitle
setproctitle("alphazero")

In [3]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

2025-01-22 23:04:39.823721: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-22 23:04:39.823754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-22 23:04:39.824897: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-22 23:04:39.830510: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-22 23:04:40.598305: W tensorflow/compiler/tf2

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import numpy as np
from tqdm import trange
from UltimateTicTacToeEnvSelfPlay import UltimateTicTacToeEnvSelfPlay
from AlphaZeroImproved import AlphaZero
import matplotlib.pyplot as plt

In [ ]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
alphazero_agent = AlphaZero(envs, state_space_shape, action_space_size, learning_rate=0.0003, simulations=200)
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    # AlphaZero turn
    alphazero_actions = alphazero_agent.play()
    alphazero_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, alphazero_reward[i], game_finished[i], _  = envs[i].step(alphazero_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = alphazero_reward[i]
    
    # A2C turn
    states = np.array([env.to_state()[0] for env in envs])
    available_actions = np.array([env.to_state()[1] for env in envs])
    actions = np.full(available_actions.shape[0], -1, dtype=int)
    one_indices = available_actions == 1
    # For each row where there is at least one '1', select a random index of '1'
    for i in range(available_actions.shape[0]):
        valid_indices = np.where(one_indices[i])[0]
        if valid_indices.size > 0:
            actions[i] = np.random.choice(valid_indices)
    reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, reward[i], game_finished[i], _  = envs[i].step(actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -reward[i]
    alphazero_agent.update_tree_with_move(alphazero_actions)
    alphazero_agent.update_tree_with_move(actions)
    print("Both players have done a move.")

win_as_first_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_first_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("AlphaZero win rate: ", win_as_first_player)
print("AlphaZero draw rate: ", draw_as_first_player)

2025-01-22 23:04:42.819512: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-01-22 23:04:42.819817: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-01-22 23:04:42.820052: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
2  out of  500
Both players have done a move.
5  out of  500
Both players have done a move.
9  out of  500
Both players have done a move.
14  out of  500
Both players have done a move.
28  out of  500
Both players have done a move.
42  out of  500
Both players have done a move.
71  out of  500
Both players have done a move.
100  out of  500
Both players 

In [ ]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    
    # A2C turn
    states = np.array([env.to_state()[0] for env in envs])
    available_actions = np.array([env.to_state()[1] for env in envs])
    actions = np.full(available_actions.shape[0], -1, dtype=int)
    one_indices = available_actions == 1
    # For each row where there is at least one '1', select a random index of '1'
    for i in range(available_actions.shape[0]):
        valid_indices = np.where(one_indices[i])[0]
        if valid_indices.size > 0:
            actions[i] = np.random.choice(valid_indices)
    reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, reward[i], game_finished[i], _  = envs[i].step(actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -reward[i]

    # AlphaZero turn
    # if 'alphazero_agent' not in locals():
    alphazero_agent = AlphaZero(envs, state_space_shape, action_space_size, learning_rate=0.0003, simulations=200)
    alphazero_actions = alphazero_agent.play()
    alphazero_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, alphazero_reward[i], game_finished[i], _  = envs[i].step(alphazero_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = alphazero_reward[i]

    alphazero_agent.update_tree_with_move(alphazero_actions)
    print("Both players have done a move.")

win_as_second_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_second_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("AlphaZero win rate: ", win_as_second_player)
print("AlphaZero draw rate: ", draw_as_second_player)

0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
5  out of  500
Both players have done a move.
9  out of  500
Both players have done a move.
15  out of  500
Both players have done a move.
30  out of  500
Both players have done a move.
47  out of  500
Both players have done a move.
67  out of  500
Both players have done a move.
101  out of  500
Both players have done a move.
136  out of  500
Both player

In [8]:
total_win = (win_as_first_player+win_as_second_player)/2
total_draw = (draw_as_first_player+draw_as_second_player)/2
print("Total win rate: ", total_win)
print("Total draw rate: ", total_draw)
print("Total loss: ", 1-total_draw-total_win)

Total win rate:  0.875
Total draw rate:  0.07300000000000001
Total loss:  0.052000000000000046
